In [0]:
DSPHS = ['bootes', 'ursaminor', 'draco']
M31 = ['m31_E0', 'm31_E1', 'm31_W0', 'm31_W1', 'm31_GSS0', 'm31_NWS0']

CONFIG_FILES = ['/home/enk/pfs/ga_targeting/configs/netflow/SSP/dSph/_common.py',
                '/home/enk/pfs/ga_targeting/configs/netflow/SSP/m31/_common.py']
for field in DSPHS:
    CONFIG_FILES.append(f'/home/enk/pfs/ga_targeting/configs/netflow/SSP/dSph/{field}.py')

# Imports and preliminaries

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pfs.ga.targeting.targets.dsph import GALAXIES as DSPH_FIELDS
from pfs.ga.targeting.targets.m31 import M31_SECTORS
from pfs.ga.targeting.instrument import *
#from pfs.ga.targeting import config
from pfs.ga.common.diagram import FOV
from pfs.ga.common.projection import WcsProjection
from astropy.time import Time
import logging

import warnings
from astropy.utils.exceptions import AstropyWarning

warnings.filterwarnings('ignore', category=AstropyWarning)

In [0]:
plt.rc('font', size=6)
cmap = plt.get_cmap('tab10')

# Cobra_charmer requires a valid logging handler
logging.getLogger().addHandler(logging.NullHandler())

obs_time = Time('2026-01-12T10:00:00')        # midnight in Hawaii; This variable needs to be defined, but it does not need to be set to any particular date.


# Load the SSP observation logs

In [0]:
from pfs.ga.targeting.targets.read_obslog_csvs import read_all_obslog_csvs

# Read all CSV files into a single dataframe
obslogs = read_all_obslog_csvs()

# Define the function that processes a single field

In [0]:
def get_ga_obslog(pnames, pointings, obslogs):
    """
    Create a DataFrame with the observation logs for each pointing.
    """
    ga_obslog = pd.DataFrame()
    ga_obslog['pointing_name'] = pnames
    ga_obslog['ra'] = [pp.ra for pp in pointings]
    ga_obslog['dec'] = [pp.dec for pp in pointings]
    ga_obslog['tot_exptime'] = [0.0 for pp in pointings]
    ga_obslog['tot_eet_b'] = [0.0 for pp in pointings]
    ga_obslog['tot_eet_r'] = [0.0 for pp in pointings]
    ga_obslog['tot_eet_m'] = [0.0 for pp in pointings]
    ga_obslog['tot_eet_n'] = [0.0 for pp in pointings]
    ga_obslog['avg_seeing'] = [0.0 for pp in pointings]
    ga_obslog['avg_transparency'] = [0.0 for pp in pointings]
    ga_obslog['dates'] = [[] for pp in pointings]
    ga_obslog['avg_exptime'] = [[] for pp in pointings]
    ga_obslog['eet_b'] = [[] for pp in pointings]
    ga_obslog['eet_r'] = [[] for pp in pointings]
    ga_obslog['eet_m'] = [[] for pp in pointings]
    ga_obslog['eet_n'] = [[] for pp in pointings]
    ga_obslog['seeing_median'] = [[] for pp in pointings]
    ga_obslog['transparency_median'] = [[] for pp in pointings]
    ga_obslog['visit_id'] = [[] for pp in pointings]
    ga_obslog['design_id'] = [[] for pp in pointings]


    for pname in ga_obslog['pointing_name']:
        galoc = ga_obslog.index[ga_obslog['pointing_name'] == pname][0]
        observation_mask = obslogs['sequence_name'].str.startswith(pname, na=False)
        observations = obslogs[observation_mask].index
        for observation in observations:
            ga_obslog.loc[galoc, 'dates'].append(obslogs.loc[observation, 'issued_at'])
            if obslogs.loc[observation, 'avg_exptime'] > 0:
                ga_obslog.loc[galoc, 'tot_exptime'] += obslogs.loc[observation, 'avg_exptime']        
            if obslogs.loc[observation, 'eet_b'] > 0:
                ga_obslog.loc[galoc, 'tot_eet_b'] += obslogs.loc[observation, 'eet_b']
            if obslogs.loc[observation, 'eet_r'] > 0:
                ga_obslog.loc[galoc, 'tot_eet_r'] += obslogs.loc[observation, 'eet_r']
            if obslogs.loc[observation, 'eet_m'] > 0:
                ga_obslog.loc[galoc, 'tot_eet_m'] += obslogs.loc[observation, 'eet_m']
            if obslogs.loc[observation, 'eet_n'] > 0:
                ga_obslog.loc[galoc, 'tot_eet_n'] += obslogs.loc[observation, 'eet_n']
            ga_obslog.loc[galoc, 'avg_exptime'].append(float(obslogs.loc[observation, 'avg_exptime']))
            ga_obslog.loc[galoc, 'eet_b'].append(float(obslogs.loc[observation, 'eet_b']))
            ga_obslog.loc[galoc, 'eet_r'].append(float(obslogs.loc[observation, 'eet_r']))
            ga_obslog.loc[galoc, 'eet_m'].append(float(obslogs.loc[observation, 'eet_m']))
            ga_obslog.loc[galoc, 'eet_n'].append(float(obslogs.loc[observation, 'eet_n']))
            ga_obslog.loc[galoc, 'seeing_median'].append(float(obslogs.loc[observation, 'seeing_median']))
            ga_obslog.loc[galoc, 'transparency_median'].append(float(obslogs.loc[observation, 'transparency_median']))
            ga_obslog.loc[galoc, 'visit_id'].append(obslogs.loc[observation, 'visit_id'])
            ga_obslog.loc[galoc, 'design_id'].append(obslogs.loc[observation, 'source_file'])
        seeing_masked = np.ma.masked_array(ga_obslog.loc[galoc, 'seeing_median'], mask=np.isnan(ga_obslog.loc[galoc, 'seeing_median']))
        weights_masked = np.ma.masked_array(ga_obslog.loc[galoc, 'avg_exptime'], mask=np.isnan(ga_obslog.loc[galoc, 'seeing_median']))
        ga_obslog.loc[galoc, 'avg_seeing'] = np.ma.average(seeing_masked, weights=weights_masked)
        transparency_masked = np.ma.masked_array(ga_obslog.loc[galoc, 'transparency_median'], mask=np.isnan(ga_obslog.loc[galoc, 'transparency_median']))
        weights_masked = np.ma.masked_array(ga_obslog.loc[galoc, 'avg_exptime'], mask=np.isnan(ga_obslog.loc[galoc, 'transparency_median']))
        ga_obslog.loc[galoc, 'avg_transparency'] = np.ma.average(transparency_masked, weights=weights_masked)
    return ga_obslog

In [0]:
def plot_field(field, pointings, pnames, ga_obslog):
    config = field.get_netflow_config()
    config.load(CONFIG_FILES, ignore_collisions=True)

    pointing = field.get_center()
    wcs = WcsProjection(pointing, proj='TAN')
    fov = FOV(projection=wcs)
    f, ax = plt.subplots(1, 1, figsize=(2.3, 2.3), dpi=240, subplot_kw=dict(projection=wcs.wcs))
    f.subplots_adjust(left=0.2, right=0.95, top=0.9, bottom=0.1)

    alpha = 0.2
    s = np.s_[::10]

    for pp, pname in zip(pointings, pnames):
        pp.obs_time = obs_time
        wfc = SubaruWFC(pp)
        pfi = SubaruPFI(wfc, instrument_options=config.instrument_options)

        pfi.plot_focal_plane(ax, fov, corners=True, c='k', ls='-', lw=0.75, label=f'Stage {pp.stage}')

        if ga_obslog.loc[ga_obslog['pointing_name'] == pname, 'tot_exptime'].values[0] > 0:
            dates = pd.to_datetime(ga_obslog.loc[ga_obslog['pointing_name'] == pname, 'dates'].values[0])
            max_diff = (np.max(dates) - np.min(dates)).days
            if max_diff > 45:
                fillcolor = 'pink'
            else:
                fillcolor = 'g'
            pfi.plot_focal_plane(ax, fov, corners=True, c='k', ls='-', lw=0.75, label=f'Stage {pp.stage}', fill=True, fillcolor=fillcolor, fillalpha=0.3)


    # Plot the repeated pointings in red
    for pp in pointings:
        if field.name == 'Ursa Minor' and pp.stage < 2 or \
        field.name == 'Draco' and pp.stage < 1 or \
        field.name == 'Sextans' and pp.stage < 3 or \
        field.name == 'Bootes I' or \
        field.name == 'Sculptor' and pp.stage < 1 or \
        field.name == 'NGC 6822' and pp.stage < 2 or \
        field.name == 'Fornax' and pp.stage < 1:
            pass
        else:
            continue

        pp.obs_time = obs_time
        wfc = SubaruWFC(pp)
        pfi = SubaruPFI(wfc, instrument_options=config.instrument_options)

        pfi.plot_focal_plane(ax, fov, corners=True, c='r', ls='-', lw=0.75, label=f'Stage {pp.stage}')        

    ax.invert_xaxis()

    # Move all ticks inside
    ax.tick_params(direction="in") #, top=True, right=True)

    # ax.grid()
    # ax.legend()

    # f.tight_layout()

    if 'M31' in field.name:
        ax.set_xlim(-10, 10)
        ax.set_ylim(-10, 10)
        ax.set_title(field.sector, fontsize=6)
    else:
        ax.set_xlim(-3, 1)
        ax.set_ylim(-3, 1)
        ax.set_title(field.name, fontsize=6)

    # f.savefig(f'dsph_pointings_{FIELD}.pdf') #, bbox_inches="tight")

In [0]:
def run_pointing(field, obslogs):
    # Load the netflow configuration for the field

    pointings = field.get_pointings(SubaruPFI)

    # Match the pointings to the obslogs
    pnames = []
    stages = [pp.stage for pp in pointings]
    pstages = {}
    for stage in np.unique(stages):
        pstages[stage] = 0
    for pp in pointings:
        if field.ID == 'm31':
            pnames.append(f"SSP_GA_{field.sector}_S{pp.stage:01d}P{pstages[pp.stage]:02d}")
        else:
            pnames.append(f"SSP_GA_{field.ID}_S{pp.stage:01d}P{pstages[pp.stage]:02d}")
        pstages[pp.stage] += 1

    ga_obslog = get_ga_obslog(pnames, pointings, obslogs)

    plot_field(field, pointings, pnames, ga_obslog)

    return ga_obslog


In [0]:
ga_obslogs = None
FIELDS = DSPHS + M31
for FIELD in FIELDS:
    if FIELD in DSPH_FIELDS:
        field = DSPH_FIELDS[FIELD]
    else:
        field = M31_SECTORS[FIELD]
    print(field.name)
    ga_obslog = run_pointing(field, obslogs)

    if ga_obslogs is not None:
        ga_obslogs = pd.concat([ga_obslogs, ga_obslog], ignore_index=True)
    else:
        ga_obslogs = ga_obslog

ga_obslogs.head()

# Export to HTML

Export the dataframe and all plots to a formatted HTML file.

In [0]:
import io
import base64
from datetime import datetime

# Re-run the loop to capture figures
figures = []
ga_obslogs = None

for FIELD in FIELDS:
    if FIELD in DSPH_FIELDS:
        field = DSPH_FIELDS[FIELD]
    else:
        field = M31_SECTORS[FIELD]
    print(f"Processing {field.name}...")
    
    # Run the pointing analysis
    ga_obslog = run_pointing(field, obslogs)
    
    # Capture the current figure
    fig = plt.gcf()
    
    # Convert figure to base64 image
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=240, bbox_inches='tight')
    buf.seek(0)
    img_base64 = base64.b64encode(buf.read()).decode('utf-8')
    buf.close()
    
    figures.append({
        'name': field.name,
        'img': img_base64
    })
    
    plt.close(fig)
    
    # Aggregate the obslog
    if ga_obslogs is not None:
        ga_obslogs = pd.concat([ga_obslogs, ga_obslog], ignore_index=True)
    else:
        ga_obslogs = ga_obslog

print(f"Captured {len(figures)} plots")

In [0]:
# Create a copy of the dataframe for display with formatted dates
ga_obslogs_display = ga_obslogs.copy()

# Convert list columns to string representations for HTML display
for col in ['dates', 'avg_exptime', 'eet_b', 'eet_r', 'eet_m', 'eet_n', 
            'seeing_median', 'transparency_median', 'visit_id', 'design_id']:
    if col in ga_obslogs_display.columns:
        ga_obslogs_display[col] = ga_obslogs_display[col].astype(str)

# Round numeric columns for cleaner display
numeric_cols = ['ra', 'dec', 'tot_exptime', 'tot_eet_b', 'tot_eet_r', 
                'tot_eet_m', 'tot_eet_n', 'avg_seeing', 'avg_transparency']
for col in numeric_cols:
    if col in ga_obslogs_display.columns:
        ga_obslogs_display[col] = ga_obslogs_display[col].round(2)

ga_obslogs_display.head()

In [0]:
# Generate HTML report
html_content = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>GA Observation Log Report</title>
    <style>
        body {{
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif;
            max-width: 1400px;
            margin: 0 auto;
            padding: 20px;
            background-color: #f5f5f5;
        }}
        h1, h2 {{
            color: #333;
            border-bottom: 2px solid #4CAF50;
            padding-bottom: 10px;
        }}
        .header {{
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 30px;
            border-radius: 10px;
            margin-bottom: 30px;
            box-shadow: 0 4px 6px rgba(0,0,0,0.1);
        }}
        .header h1 {{
            margin: 0;
            border: none;
            color: white;
        }}
        .timestamp {{
            font-size: 0.9em;
            opacity: 0.9;
            margin-top: 10px;
        }}
        .section {{
            background: white;
            padding: 25px;
            margin-bottom: 30px;
            border-radius: 8px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }}
        table {{
            border-collapse: collapse;
            width: 100%;
            margin: 20px 0;
            font-size: 0.9em;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }}
        th {{
            background-color: #4CAF50;
            color: white;
            text-align: left;
            padding: 12px;
            font-weight: 600;
            position: sticky;
            top: 0;
        }}
        td {{
            padding: 10px 12px;
            border-bottom: 1px solid #ddd;
        }}
        tr:hover {{
            background-color: #f5f5f5;
        }}
        tr:nth-child(even) {{
            background-color: #f9f9f9;
        }}
        .plot-container {{
            display: flex;
            flex-wrap: wrap;
            gap: 30px;
            justify-content: center;
            margin: 20px 0;
        }}
        .plot-item {{
            flex: 0 1 auto;
            text-align: center;
            background: white;
            padding: 20px;
            border-radius: 8px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }}
        .plot-item h3 {{
            margin-top: 0;
            margin-bottom: 15px;
            color: #555;
        }}
        .plot-item img {{
            max-width: 100%;
            height: auto;
            border: 1px solid #ddd;
            border-radius: 4px;
        }}
        .stats {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
            gap: 15px;
            margin: 20px 0;
        }}
        .stat-box {{
            background: #f8f9fa;
            padding: 15px;
            border-radius: 6px;
            border-left: 4px solid #4CAF50;
        }}
        .stat-label {{
            font-size: 0.85em;
            color: #666;
            text-transform: uppercase;
            letter-spacing: 0.5px;
        }}
        .stat-value {{
            font-size: 1.5em;
            font-weight: bold;
            color: #333;
            margin-top: 5px;
        }}
    </style>
</head>
<body>
    <div class="header">
        <h1>🔭 GA Observation Log Report</h1>
        <div class="timestamp">Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</div>
    </div>
    
    <div class="section">
        <h2>📊 Summary Statistics</h2>
        <div class="stats">
            <div class="stat-box">
                <div class="stat-label">Total Pointings</div>
                <div class="stat-value">{len(ga_obslogs)}</div>
            </div>
            <div class="stat-box">
                <div class="stat-label">Total Fields</div>
                <div class="stat-value">{len(FIELDS)}</div>
            </div>
            <div class="stat-box">
                <div class="stat-label">Total Exposure Time</div>
                <div class="stat-value">{ga_obslogs['tot_exptime'].sum():.1f}s</div>
            </div>
            <div class="stat-box">
                <div class="stat-label">Avg Seeing</div>
                <div class="stat-value">{ga_obslogs['avg_seeing'].mean():.2f}"</div>
            </div>
        </div>
    </div>
    
    <div class="section">
        <h2>🗺️ Field Plots</h2>
        <div class="plot-container">
        <p><strong>Legend:</strong>
        <ul>
            <li> Black outline: Fields intended to be osberved in one epoch.
            <li> Red outline: Fields intended to be observed in two epochs.
            <li> Green shading: Already observed in one run.
            <li> Pink shading: Already observed in multiple runs separated by at least 45 days.
        </ul>    
"""

for fig_data in figures:
    html_content += f"""
            <div class="plot-item">
                <h3>{fig_data['name']}</h3>
                <img src="data:image/png;base64,{fig_data['img']}" alt="{fig_data['name']} pointing plot">
            </div>
"""

html_content += """
        </div>
    </div>
    
    <div class="section">
        <h2>📋 Observation Log Data - Summary</h2>
        <p>This table shows summary statistics for each pointing with observations (tot_exptime > 0), excluding detailed per-observation data.</p>
"""

# Create a simplified version without list columns and filter out zero exposure time
list_columns = ['dates', 'avg_exptime', 'eet_b', 'eet_r', 'eet_m', 'eet_n', 
                'seeing_median', 'transparency_median', 'visit_id', 'design_id']
summary_cols = [col for col in ga_obslogs_display.columns if (col not in list_columns and not col.endswith('_r'))]
ga_obslogs_summary = ga_obslogs_display[ga_obslogs_display['tot_exptime'] > 0][summary_cols]
html_content += ga_obslogs_summary.to_html(index=False, classes='dataframe', border=0)

html_content += """
    </div>
    
    <div class="section">
        <h2>📋 Observation Log Data - Detailed</h2>
        <p>This table includes all columns with per-observation details (shown as lists).</p>
"""

# Filter out columns ending in '_r' for HTML display
display_cols = [col for col in ga_obslogs_display.columns if not col.endswith('_r')]
html_content += ga_obslogs_display[display_cols].to_html(index=False, classes='dataframe', border=0)

html_content += """
    </div>
</body>
</html>
"""

# Save to file
output_file = 'ga_obslog_report.html'
with open(output_file, 'w') as f:
    f.write(html_content)

print(f"✅ HTML report saved to: {output_file}")
print(f"   - {len(figures)} field plots included")
print(f"   - {len(ga_obslogs)} pointing records in table")